In [36]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, lit, month, when, count
from pyspark.sql.types import DecimalType
import os

In [37]:
#Iniciamos la sesion de spark
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Evidencia1") \
    .getOrCreate()

#Verificacion del entorno
spark.conf.set("spark.sql.eagerEval.enable",True)
spark

In [38]:
#Obtenemos todos los archivos que componen nuestra base datos
folder = r'C:\airline delay analysis'
archivos = os.listdir(folder)
archivos

['2009.csv',
 '2010.csv',
 '2011.csv',
 '2012.csv',
 '2013.csv',
 '2014.csv',
 '2015.csv',
 '2016.csv',
 '2017.csv',
 '2018.csv',
 '2019.csv']

In [39]:
#Precargamos una de las bases de datos
df = spark.read.csv(r"C:\airline delay analysis\2009.csv", header=True, inferSchema=True)

#Unimos todas las bases de datos
for fl in archivos[1:]:
    print(fl)
    dfTemp = spark.read.csv(folder+f'\\{fl}', header=True, inferSchema=True)
    df = df.unionByName(dfTemp,allowMissingColumns=True)

2010.csv
2011.csv
2012.csv
2013.csv
2014.csv
2015.csv
2016.csv
2017.csv
2018.csv
2019.csv


In [40]:
#Vista de tabla con Pandas solo para visualizacion
df.show(10)

+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+---

In [41]:
#Obtenemso el conteo de filas con las que cuenta el df
p = df.count()
print("Población (vuelos totales):", p)

Población (vuelos totales): 68979001


In [42]:
#Verificamos el esquema
df.printSchema()

root
 |-- FL_DATE: date (nullable = true)
 |-- OP_CARRIER: string (nullable = true)
 |-- OP_CARRIER_FL_NUM: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: double (nullable = true)
 |-- DEP_TIME: double (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- TAXI_OUT: double (nullable = true)
 |-- WHEELS_OFF: double (nullable = true)
 |-- WHEELS_ON: double (nullable = true)
 |-- TAXI_IN: double (nullable = true)
 |-- CRS_ARR_TIME: double (nullable = true)
 |-- ARR_TIME: double (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- CANCELLATION_CODE: string (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- CRS_ELAPSED_TIME: double (nullable = true)
 |-- ACTUAL_ELAPSED_TIME: double (nullable = true)
 |-- AIR_TIME: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double

In [43]:
#Creamos columna mes
df = df.withColumn("month", month("FL_DATE"))

#Verificamos los datos de nuevo
df.show(10) 

+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+-----+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|month|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+-

In [45]:
#Generamos la columna estacion en referencia a la estacion del año, creando asi la primer particion
df = df.withColumn(
    "season",
    when(col("month").isin(12, 1, 2), "Invierno")
    .when(col("month").isin(3, 4, 5), "Primavera")
    .when(col("month").isin(6, 7, 8), "Verano")
    .otherwise("Otoño")
)

#Verificamos la creacion de las estaciones
df.show(10)

+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-------------+-------------+---------+--------------+-------------------+-----------+-----------------+----+-----+--------+
|   FL_DATE|OP_CARRIER|OP_CARRIER_FL_NUM|ORIGIN|DEST|CRS_DEP_TIME|DEP_TIME|DEP_DELAY|TAXI_OUT|WHEELS_OFF|WHEELS_ON|TAXI_IN|CRS_ARR_TIME|ARR_TIME|ARR_DELAY|CANCELLED|CANCELLATION_CODE|DIVERTED|CRS_ELAPSED_TIME|ACTUAL_ELAPSED_TIME|AIR_TIME|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|Unnamed: 27|OP_UNIQUE_CARRIER|_c20|month|  season|
+----------+----------+-----------------+------+----+------------+--------+---------+--------+----------+---------+-------+------------+--------+---------+---------+-----------------+--------+----------------+-------------------+--------+--------+-----------

In [46]:
#Realizamos una particion mas ahora verificando los retrasos registrados
tabla_retrasos = (
    df.groupBy("season").agg(
        count("DEP_DELAY").alias("vuelos_totales"), #Cantidad de vuelos por estacion
        sum(when(col("DEP_DELAY") > 0, 1).otherwise(0)).alias("vuelos_retrasados"), #Cantidad de vuelos retrasados
        sum(when(col("CARRIER_DELAY") > 0, 1).otherwise(0)).alias("retrasos_operacion"),#Cantidad de vuelos por motivo de operacion (mantenimiento, traslado, estacionamiento, etc.)
        sum(when(col("WEATHER_DELAY") > 0, 1).otherwise(0)).alias("retrasos_clima") #Cantidad de vuelos retrasados por clima
    )
)

tabla_retrasos.show(10)

+---------+--------------+-----------------+------------------+--------------+
|   season|vuelos_totales|vuelos_retrasados|retrasos_operacion|retrasos_clima|
+---------+--------------+-----------------+------------------+--------------+
| Invierno|      15921902|          6074316|           1618356|        207557|
|Primavera|      17291594|          6223481|           1570982|        167828|
|   Verano|      17967686|          7261065|           1954122|        243290|
|    Otoño|      16727034|          5266184|           1226581|        101033|
+---------+--------------+-----------------+------------------+--------------+



In [48]:
#Creamos las posibilidades de cada caso
tabla_probabilidades = (
    tabla_retrasos
    .withColumn("prob_temporada", (col("vuelos_totales") / lit(p)).cast(DecimalType(3,3))) #Probabilidad de que un vuelo ocurra en cierta estacion.
    .withColumn("prob_retraso_salida", (col("vuelos_retrasados") / col("vuelos_totales")).cast(DecimalType(3,3))) #Probalidad de sufrir un retraso en cierta estacion
    .withColumn("prob_retraso_operacion", (col("retrasos_operacion") / col("vuelos_totales")).cast(DecimalType(3,3))) #Probalidad de que retraso sea por operacion
    .withColumn("prob_retraso_clima", (col("retrasos_clima") / col("vuelos_totales")).cast(DecimalType(3,3))) #Probabilidad de que retraso sea por clima
    .select(
        "season",
        "prob_temporada",
        "prob_retraso_salida",
        "prob_retraso_operacion",
        "prob_retraso_clima"
    )
)

tabla_probabilidades.show(10)    


+---------+--------------+-------------------+----------------------+------------------+
|   season|prob_temporada|prob_retraso_salida|prob_retraso_operacion|prob_retraso_clima|
+---------+--------------+-------------------+----------------------+------------------+
| Invierno|         0.231|              0.382|                 0.102|             0.013|
|Primavera|         0.251|              0.360|                 0.091|             0.010|
|   Verano|         0.260|              0.404|                 0.109|             0.014|
|    Otoño|         0.242|              0.315|                 0.073|             0.006|
+---------+--------------+-------------------+----------------------+------------------+

